# HF Multimodel Notebook: Config-Driven Quantization

This notebook uses a centralized `MODEL_CONFIGS` list and a universal runner.
It supports per-model quantization, generation, thinking control, and optional compatibility hooks.


In [ ]:
# Install/update once if needed:
# If architecture support is missing in your transformers build:
%pip install -U transformers accelerate bitsandbytes huggingface_hub pandas
%pip install -U git+https://github.com/huggingface/transformers.git
# Restart kernel after updates.

In [ ]:
from __future__ import annotations

import gc
import inspect
import os
import re
import time
import warnings
from copy import deepcopy
from dataclasses import asdict, dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import torch
import transformers
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    from transformers import BitsAndBytesConfig
except Exception:
    BitsAndBytesConfig = None

HF_TOKEN = os.getenv("HF_TOKEN", "").strip()


def get_runtime_info() -> Dict[str, Any]:
    return {
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "bitsandbytes_available": BitsAndBytesConfig is not None,
        "hf_token_loaded": bool(HF_TOKEN),
    }


runtime_info = get_runtime_info()
display(pd.DataFrame([runtime_info]))

if not runtime_info["hf_token_loaded"]:
    warnings.warn("HF_TOKEN is not set. Gated models may fail with auth errors.")

In [ ]:
# ---------- Types ----------
@dataclass
class QuantConfig:
    mode: str = "none"  # none, fp16, bf16, int8, int4_nf4, int4_fp4
    bnb_4bit_quant_type: str = "nf4"
    bnb_4bit_compute_dtype: str = "bf16"
    bnb_4bit_use_double_quant: bool = True


@dataclass
class GenerationConfig:
    max_new_tokens: int = 128
    do_sample: bool = True
    temperature: float = 0.2
    top_p: float = 0.95
    repetition_penalty: float = 1.0


@dataclass
class CompatHooks:
    phi_losskwargs_shim: bool = False
    gemma_strict_no_thinking: bool = False


@dataclass
class ModelConfig:
    name: str
    repo_id: str
    enabled: bool = True
    device_map: str = "auto"
    trust_remote_code: bool = True
    dtype_mode: str = "auto"  # auto, fp16, bf16, fp32
    quant: QuantConfig = field(default_factory=QuantConfig)
    disable_think: bool = True
    compat_hooks: CompatHooks = field(default_factory=CompatHooks)
    generation_override: Optional[GenerationConfig] = None


@dataclass
class LLMRequest:
    system_prompt: str
    user_prompt: str
    generation_default: GenerationConfig = field(default_factory=GenerationConfig)


@dataclass
class InferenceResult:
    model_name: str
    repo_id: str
    status: str
    latency_sec: float
    runtime_device: str
    quant_mode: str
    response_text: str
    error_code: str
    error_message: Optional[str]
    resolution_hint: str

In [ ]:
# ---------- Config ----------
MODEL_CONFIGS: List[ModelConfig] = [
    ModelConfig(
        name="Qwen2.5-Coder-7B-Instruct",
        repo_id="Qwen/Qwen2.5-Coder-7B-Instruct",
        device_map="auto",
        trust_remote_code=True,
        dtype_mode="auto",
        quant=QuantConfig(mode="int4_nf4", bnb_4bit_compute_dtype="bf16"),
        disable_think=True,
    ),
    ModelConfig(
        name="Qwen3.5-9B",
        repo_id="Qwen/Qwen3.5-9B",
        device_map="auto",
        trust_remote_code=True,
        dtype_mode="bf16",
        quant=QuantConfig(mode="none"),
        disable_think=True,
    ),
    ModelConfig(
        name="Qwen3-8B",
        repo_id="Qwen/Qwen3-8B",
        device_map="auto",
        trust_remote_code=True,
        dtype_mode="auto",
        quant=QuantConfig(mode="int4_nf4", bnb_4bit_compute_dtype="bf16"),
        disable_think=True,
    ),
    ModelConfig(
        name="gemma-3-12b-it",
        repo_id="google/gemma-3-12b-it",
        device_map="auto",
        trust_remote_code=True,
        dtype_mode="bf16",
        quant=QuantConfig(mode="none"),
        disable_think=True,
        compat_hooks=CompatHooks(gemma_strict_no_thinking=True),
    ),
    ModelConfig(
        name="gemma-4-E4B-it",
        repo_id="google/gemma-4-E4B-it",
        device_map="auto",
        trust_remote_code=True,
        dtype_mode="bf16",
        quant=QuantConfig(mode="none"),
        disable_think=True,
        compat_hooks=CompatHooks(gemma_strict_no_thinking=True),
    ),
    ModelConfig(
        name="Phi-4-mini-instruct",
        repo_id="microsoft/Phi-4-mini-instruct",
        device_map="auto",
        trust_remote_code=False,
        dtype_mode="bf16",
        quant=QuantConfig(mode="none"),
        disable_think=True,
        compat_hooks=CompatHooks(phi_losskwargs_shim=True),
    ),
]

REQUEST = LLMRequest(
    system_prompt="You are a helpful assistant. Follow the user instruction exactly.",
    user_prompt="Write a short greeting in Russian.",
    generation_default=GenerationConfig(
        max_new_tokens=96,
        do_sample=True,
        temperature=0.2,
        top_p=0.95,
        repetition_penalty=1.0,
    ),
)

display(
    pd.DataFrame(
        [
            {
                "name": m.name,
                "repo_id": m.repo_id,
                "enabled": m.enabled,
                "quant_mode": m.quant.mode,
                "dtype_mode": m.dtype_mode,
                "device_map": m.device_map,
                "disable_think": m.disable_think,
                "phi_hook": m.compat_hooks.phi_losskwargs_shim,
                "gemma_hook": m.compat_hooks.gemma_strict_no_thinking,
            }
            for m in MODEL_CONFIGS
        ]
    )
)

In [ ]:
# ---------- Validation ----------
ALLOWED_QUANT_MODES = {"none", "fp16", "bf16", "int8", "int4_nf4", "int4_fp4"}
ALLOWED_DTYPE_MODES = {"auto", "fp16", "bf16", "fp32"}


def validate_model_configs(
    model_configs: List[ModelConfig], runtime_info: Dict[str, Any]
) -> None:
    errors: List[str] = []

    if not model_configs:
        raise ValueError("MODEL_CONFIGS is empty.")

    seen_names: set[str] = set()

    for cfg in model_configs:
        if cfg.name in seen_names:
            errors.append(f"[{cfg.name}] duplicate model name.")
        seen_names.add(cfg.name)

        if not cfg.enabled:
            continue

        if cfg.quant.mode not in ALLOWED_QUANT_MODES:
            errors.append(f"[{cfg.name}] invalid quant.mode={cfg.quant.mode}")

        if cfg.dtype_mode not in ALLOWED_DTYPE_MODES:
            errors.append(f"[{cfg.name}] invalid dtype_mode={cfg.dtype_mode}")

        if cfg.quant.mode in {"int8", "int4_nf4", "int4_fp4"}:
            if not runtime_info["bitsandbytes_available"]:
                errors.append(
                    f"[{cfg.name}] quant mode {cfg.quant.mode} requires bitsandbytes."
                )
            if not runtime_info["cuda_available"]:
                errors.append(
                    f"[{cfg.name}] quant mode {cfg.quant.mode} requires CUDA in this notebook setup."
                )
            if cfg.dtype_mode != "auto":
                errors.append(
                    f"[{cfg.name}] dtype_mode must be auto when quant mode is {cfg.quant.mode}."
                )

        if cfg.quant.mode == "int4_nf4" and cfg.quant.bnb_4bit_quant_type != "nf4":
            errors.append(f"[{cfg.name}] int4_nf4 requires bnb_4bit_quant_type=nf4.")

        if cfg.quant.mode == "int4_fp4" and cfg.quant.bnb_4bit_quant_type != "fp4":
            errors.append(f"[{cfg.name}] int4_fp4 requires bnb_4bit_quant_type=fp4.")

    if errors:
        raise ValueError("Config validation failed:\n- " + "\n- ".join(errors))


validate_model_configs(MODEL_CONFIGS, runtime_info)
print("Config validation passed.")

In [ ]:
# ---------- Loader ----------
def _resolve_torch_dtype(dtype_mode: str) -> torch.dtype:
    if dtype_mode == "fp16":
        return torch.float16
    if dtype_mode == "bf16":
        return torch.bfloat16
    if dtype_mode == "fp32":
        return torch.float32

    if torch.cuda.is_available():
        return torch.bfloat16
    return torch.float32


def _resolve_4bit_compute_dtype(name: str) -> torch.dtype:
    mapping = {
        "bf16": torch.bfloat16,
        "fp16": torch.float16,
        "fp32": torch.float32,
    }
    if name not in mapping:
        raise ValueError(f"Unsupported bnb_4bit_compute_dtype: {name}")
    return mapping[name]


def build_quantization_config(
    model_cfg: ModelConfig, runtime_info: Dict[str, Any]
) -> Dict[str, Any]:
    mode = model_cfg.quant.mode

    if mode in {"none", "fp16", "bf16"}:
        dtype_mode = model_cfg.dtype_mode
        if mode in {"fp16", "bf16"}:
            dtype_mode = mode
        return {"torch_dtype": _resolve_torch_dtype(dtype_mode)}

    if mode == "int8":
        return {
            "quantization_config": BitsAndBytesConfig(load_in_8bit=True),
        }

    if mode in {"int4_nf4", "int4_fp4"}:
        quant_type = "nf4" if mode == "int4_nf4" else "fp4"
        return {
            "quantization_config": BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type=quant_type,
                bnb_4bit_use_double_quant=bool(
                    model_cfg.quant.bnb_4bit_use_double_quant
                ),
                bnb_4bit_compute_dtype=_resolve_4bit_compute_dtype(
                    model_cfg.quant.bnb_4bit_compute_dtype
                ),
            ),
        }

    raise ValueError(f"Unsupported quant mode: {mode}")


def _apply_phi_losskwargs_shim() -> bool:
    try:
        from typing import TypedDict
        import transformers.utils as tf_utils

        if hasattr(tf_utils, "LossKwargs"):
            return False

        class LossKwargs(TypedDict, total=False):
            pass

        tf_utils.LossKwargs = LossKwargs
        return True
    except Exception:
        return False


def load_tokenizer(model_cfg: ModelConfig, auth: Dict[str, Any]) -> AutoTokenizer:
    tokenizer = AutoTokenizer.from_pretrained(
        model_cfg.repo_id,
        trust_remote_code=model_cfg.trust_remote_code,
        **auth,
    )

    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    return tokenizer


def load_model(
    model_cfg: ModelConfig, auth: Dict[str, Any], runtime_info: Dict[str, Any]
) -> Tuple[Any, Dict[str, Any]]:
    if model_cfg.compat_hooks.phi_losskwargs_shim:
        _ = _apply_phi_losskwargs_shim()

    model_kwargs: Dict[str, Any] = {
        "trust_remote_code": model_cfg.trust_remote_code,
        "device_map": model_cfg.device_map,
        **auth,
    }
    model_kwargs.update(build_quantization_config(model_cfg, runtime_info))

    model = AutoModelForCausalLM.from_pretrained(model_cfg.repo_id, **model_kwargs)

    runtime_device = "unknown"
    try:
        runtime_device = str(next(model.parameters()).device)
    except Exception:
        runtime_device = str(model_cfg.device_map)

    runtime_meta = {
        "runtime_device": runtime_device,
        "quant_mode": model_cfg.quant.mode,
        "trust_remote_code": model_cfg.trust_remote_code,
    }
    return model, runtime_meta


def unload_resources(model: Any, tokenizer: Any) -> None:
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# ---------- Prompt / Inference ----------
def _merge_generation_config(
    request: LLMRequest, model_cfg: ModelConfig
) -> GenerationConfig:
    merged = deepcopy(request.generation_default)
    if model_cfg.generation_override is not None:
        override_data = asdict(model_cfg.generation_override)
        for key, value in override_data.items():
            setattr(merged, key, value)
    return merged


def _is_qwen_repo(repo_id: str) -> bool:
    return str(repo_id).lower().startswith("qwen/")


def _extract_final_tag(text: str) -> str:
    match = re.search(r"(?is)<final>\s*(.*?)\s*</final>", text or "")
    if not match:
        return ""
    return match.group(1).strip()


def _strip_reasoning_text(text: str) -> str:
    raw = (text or "").strip()
    if not raw:
        return ""

    tagged = _extract_final_tag(raw)
    if tagged:
        return tagged

    cleaned = re.sub(r"(?is)<think>.*?</think>", "", raw)
    cleaned = cleaned.replace("<think>", "").replace("</think>", "").strip()

    final_patterns = [
        r"(?is)(?:final answer|answer)\s*[:\-]\s*(.+)$",
        r"(?is)(?:итог|ответ)\s*[:\-]\s*(.+)$",
    ]
    for pattern in final_patterns:
        match = re.search(pattern, cleaned)
        if match:
            candidate = match.group(1).strip()
            if candidate:
                return candidate

    markers = ["thinking process:", "thought process:", "reasoning:", "analysis:", "analyze the request"]
    lower = cleaned.lower()

    for marker in markers:
        idx = lower.find(marker)
        if idx > 0:
            candidate = cleaned[:idx].strip()
            if candidate:
                return candidate
        elif idx == 0:
            tail = cleaned[len(marker):].strip()
            tail = re.sub(r"(?is)^[-*\d\.)\s:]+", "", tail).strip()
            if not tail:
                return ""

            sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", tail) if s.strip()]
            for sent in sentences:
                sent_lower = sent.lower()
                if any(k in sent_lower for k in ["thinking", "analysis", "step", "task", "analyze", "шаг", "анализ"]):
                    continue
                return sent

            return ""

    return cleaned


def _looks_like_reasoning_only(text: str) -> bool:
    value = (text or "").strip().lower()
    if not value:
        return True

    reasoning_markers = [
        "thinking process",
        "thought process",
        "analysis",
        "reasoning",
        "analyze the request",
        "task:",
        "step",
    ]
    if any(m in value for m in reasoning_markers):
        greeting_markers = ["привет", "здравств", "добрый", "hello", "hi"]
        if not any(g in value for g in greeting_markers):
            return True

    return False


def _render_chat_template(
    tokenizer: AutoTokenizer,
    messages: List[Dict[str, str]],
    base_kwargs: Dict[str, Any],
    model_cfg: ModelConfig,
) -> str:
    attempts: List[Dict[str, Any]] = []
    attempts.append(dict(base_kwargs))

    if model_cfg.disable_think:
        # Generic path for tokenizers exposing `enable_thinking` directly.
        try:
            sig = inspect.signature(tokenizer.apply_chat_template)
            if "enable_thinking" in sig.parameters:
                kw = dict(base_kwargs)
                kw["enable_thinking"] = False
                attempts.insert(0, kw)
        except Exception:
            pass

        # Qwen-compatible paths (mirrors API style: chat_template_kwargs).
        if _is_qwen_repo(model_cfg.repo_id):
            kw1 = dict(base_kwargs)
            kw1["enable_thinking"] = False
            attempts.insert(0, kw1)

            kw2 = dict(base_kwargs)
            kw2["chat_template_kwargs"] = {"enable_thinking": False}
            attempts.insert(0, kw2)

            kw3 = dict(base_kwargs)
            kw3["enable_thinking"] = False
            kw3["chat_template_kwargs"] = {"enable_thinking": False}
            attempts.insert(0, kw3)

    last_exc: Optional[Exception] = None
    for kwargs in attempts:
        try:
            return tokenizer.apply_chat_template(messages, **kwargs)
        except Exception as exc:
            last_exc = exc
            continue

    if last_exc is not None:
        raise last_exc
    raise RuntimeError("Unable to render chat template.")


def build_prompt(
    tokenizer: AutoTokenizer, request: LLMRequest, model_cfg: ModelConfig
) -> str:
    system_prompt = request.system_prompt

    if model_cfg.disable_think:
        system_prompt += (
            "\n\nReturn only the final answer. Do not output reasoning traces."
            "\nOutput format is mandatory: <final>YOUR_ANSWER</final>."
        )

    if model_cfg.compat_hooks.gemma_strict_no_thinking:
        system_prompt += (
            "\n\nStrict output rules:"
            "\n- Output only final answer text."
            "\n- Never output Thinking Process/analysis/steps."
            "\n- No explanations."
        )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": request.user_prompt},
    ]

    if hasattr(tokenizer, "apply_chat_template"):
        base_kwargs: Dict[str, Any] = {"tokenize": False, "add_generation_prompt": True}
        try:
            return _render_chat_template(tokenizer, messages, base_kwargs, model_cfg)
        except Exception:
            pass

    return (
        "System:\n"
        + system_prompt
        + "\n\n"
        + "User:\n"
        + request.user_prompt
        + "\n\n"
        + "Assistant:\n"
    )


def _build_retry_prompt(tokenizer: AutoTokenizer, request: LLMRequest, model_cfg: ModelConfig) -> str:
    retry_system = (
        "Output exactly one short final answer in the user's language. "
        "No analysis, no steps, no reasoning. "
        "Mandatory format: <final>YOUR_ANSWER</final>."
    )
    messages = [
        {"role": "system", "content": retry_system},
        {"role": "user", "content": request.user_prompt},
    ]

    if hasattr(tokenizer, "apply_chat_template"):
        base_kwargs: Dict[str, Any] = {"tokenize": False, "add_generation_prompt": True}
        try:
            return _render_chat_template(tokenizer, messages, base_kwargs, model_cfg)
        except Exception:
            pass

    return "System:\n" + retry_system + "\n\nUser:\n" + request.user_prompt + "\n\nAssistant:\n"


def _decode_generation(model: Any, tokenizer: AutoTokenizer, prompt: str, gen_kwargs: Dict[str, Any]) -> str:
    inputs = tokenizer(prompt, return_tensors="pt")
    target_device = next(model.parameters()).device
    inputs = {k: v.to(target_device) for k, v in inputs.items()}

    with torch.inference_mode():
        output_ids = model.generate(**inputs, **gen_kwargs)

    prompt_tokens = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][prompt_tokens:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def generate_response(
    model: Any,
    tokenizer: AutoTokenizer,
    prompt: str,
    model_cfg: ModelConfig,
    request: LLMRequest,
) -> str:
    gen_cfg = _merge_generation_config(request, model_cfg)

    do_sample_now = bool(gen_cfg.do_sample)
    if model_cfg.compat_hooks.gemma_strict_no_thinking and model_cfg.disable_think:
        do_sample_now = False

    gen_kwargs: Dict[str, Any] = {
        "max_new_tokens": int(gen_cfg.max_new_tokens),
        "do_sample": do_sample_now,
        "repetition_penalty": float(gen_cfg.repetition_penalty),
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    if do_sample_now:
        gen_kwargs["temperature"] = float(gen_cfg.temperature)
        gen_kwargs["top_p"] = float(gen_cfg.top_p)

    raw_response = _decode_generation(model, tokenizer, prompt, gen_kwargs)

    response_text = raw_response
    if model_cfg.disable_think:
        cleaned = _strip_reasoning_text(raw_response)
        response_text = cleaned if cleaned else raw_response

    if model_cfg.disable_think and _looks_like_reasoning_only(response_text):
        retry_prompt = _build_retry_prompt(tokenizer, request, model_cfg)
        retry_gen_kwargs = {
            "max_new_tokens": max(24, min(64, int(gen_cfg.max_new_tokens))),
            "do_sample": False,
            "repetition_penalty": float(gen_cfg.repetition_penalty),
            "pad_token_id": tokenizer.pad_token_id,
            "eos_token_id": tokenizer.eos_token_id,
        }
        retry_raw = _decode_generation(model, tokenizer, retry_prompt, retry_gen_kwargs)
        retry_clean = _strip_reasoning_text(retry_raw)
        if retry_clean:
            response_text = retry_clean

    return (response_text or raw_response).strip()


In [ ]:
# ---------- Runner / Diagnostics ----------
def _classify_error(error_message: str) -> Tuple[str, str]:
    msg = (error_message or "").lower()

    if (
        "gated repo" in msg
        or "401 client error" in msg
        or "you are trying to access a gated repo" in msg
    ):
        return "auth_error", "Grant model access on HF and ensure HF_TOKEN is valid."

    if "bitsandbytes" in msg or "4bit" in msg or "8bit" in msg:
        return (
            "quant_not_available",
            "Install bitsandbytes/CUDA build and verify selected quantization mode.",
        )

    if "no kernel image is available for execution on the device" in msg:
        return (
            "cuda_mismatch",
            "Your CUDA/PyTorch build does not match the GPU architecture.",
        )

    if "does not recognize this architecture" in msg or "model type" in msg:
        return (
            "arch_unsupported",
            "Upgrade transformers to a version supporting this model architecture.",
        )

    return (
        "runtime_error",
        "Inspect full traceback and model config. This run uses strict fail policy.",
    )


def run_models(request: LLMRequest, model_configs: List[ModelConfig]) -> pd.DataFrame:
    runtime = get_runtime_info()
    validate_model_configs(model_configs, runtime)

    auth = {"token": HF_TOKEN} if HF_TOKEN else {}
    rows: List[InferenceResult] = []

    for model_cfg in model_configs:
        if not model_cfg.enabled:
            continue

        print(f"Running: {model_cfg.name} ({model_cfg.repo_id})")

        tokenizer = None
        model = None
        started = time.perf_counter()

        status = "ok"
        response_text = ""
        error_message: Optional[str] = None
        error_code = ""
        resolution_hint = ""
        runtime_device = "unknown"

        try:
            tokenizer = load_tokenizer(model_cfg, auth)
            model, runtime_meta = load_model(model_cfg, auth, runtime)
            runtime_device = str(runtime_meta.get("runtime_device", runtime_device))

            prompt = build_prompt(tokenizer, request, model_cfg)
            response_text = generate_response(
                model, tokenizer, prompt, model_cfg, request
            )
        except Exception as exc:
            status = "error"
            error_message = f"{type(exc).__name__}: {exc}"
            error_code, resolution_hint = _classify_error(error_message)
        finally:
            latency_sec = time.perf_counter() - started
            if model is not None and tokenizer is not None:
                unload_resources(model, tokenizer)
            else:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        rows.append(
            InferenceResult(
                model_name=model_cfg.name,
                repo_id=model_cfg.repo_id,
                status=status,
                latency_sec=latency_sec,
                runtime_device=runtime_device,
                quant_mode=model_cfg.quant.mode,
                response_text=response_text,
                error_code=error_code,
                error_message=error_message,
                resolution_hint=resolution_hint,
            )
        )

    result_df = pd.DataFrame([asdict(row) for row in rows])
    if not result_df.empty:
        result_df = result_df.sort_values(by=["status", "model_name"]).reset_index(
            drop=True
        )
    return result_df

In [ ]:
# ---------- Validation smoke tests ----------
runtime = get_runtime_info()

# 1) invalid quant mode
bad_quant_cfg = deepcopy(MODEL_CONFIGS)
bad_quant_cfg[0].quant.mode = "int2"
try:
    validate_model_configs(bad_quant_cfg, runtime)
    raise AssertionError("Expected validation error for invalid quant mode.")
except ValueError:
    pass

# 2) dtype conflict with int4
bad_dtype_cfg = deepcopy(MODEL_CONFIGS)
bad_dtype_cfg[0].quant.mode = "int4_nf4"
bad_dtype_cfg[0].dtype_mode = "fp16"
try:
    validate_model_configs(bad_dtype_cfg, runtime)
    raise AssertionError("Expected validation error for dtype/quant conflict.")
except ValueError:
    pass

print("Validation smoke tests passed.")

In [ ]:
# ---------- Run recipes ----------
# Smoke run (one model):
# smoke_df = run_models(REQUEST, [m for m in MODEL_CONFIGS if m.enabled][:1])
# display(smoke_df)

# Full run:
# results_df = run_models(REQUEST, MODEL_CONFIGS)
# display(results_df)

# Optional save:
# results_df.to_csv('/kaggle/working/data.csv', index=False)